In [1]:
import torch
import torch.nn as nn
import math

In [2]:
class InputEmbeddings(nn.Module):
    def __init__(self, d_model: int, vocab_size: int):
        super().__init__()
        self.d_model= d_model
        self.vocab_size= vocab_size
        self.embedding= nn.Embedding(vocab_size, d_model)
    
    def forward(self, x):
        return self.embedding(x) * math.sqrt(self.d_model)


In [3]:
d_model = 8
vocap_size = 1000
embed = InputEmbeddings(d_model, vocap_size)
sentense_tokens = torch.tensor([[10, 25, 500, 30, 31, 85]])
output = embed(sentense_tokens)
print(output.shape)

torch.Size([1, 6, 8])


In [4]:
print(output)

tensor([[[-0.3419, -3.9759,  1.6379,  6.0379, -3.9306, -1.3425,  2.7122,
           3.1752],
         [-0.5485,  0.0668, -1.1408, -3.2420, -1.2291,  3.8582,  1.3941,
           1.0094],
         [-0.0268,  1.8697,  0.3030,  0.3090, -1.4471,  3.7451,  0.3327,
           0.0647],
         [-1.5788,  2.2405, -5.2849,  0.0474,  4.5467, -0.5449, -0.3373,
           1.1731],
         [-0.3234,  1.3104, -2.2854, -2.0227, -2.1241,  2.2917,  4.4512,
           6.4036],
         [ 0.4411,  3.1247, -5.7574, -3.8035,  2.9027, -2.0492, -0.7777,
          -3.6704]]], grad_fn=<MulBackward0>)


In [5]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, seq_len: int, dropout: float):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        self.d_model = d_model
        self.seq_len = seq_len
        pe = torch.zeros(seq_len, d_model)
        postition = torch.arange(0, seq_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2]=torch.sin(postition * div_term)
        pe[:, 1::2]=torch.cos(postition * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe',pe)
    
    def forward(self, x):
        x = x +(self.pe[:, :x.shape[1], :]).requires_grad_(False)
        return self.dropout(x)


In [6]:
d_model  = 8
seq_len = 1000
pos_enc = PositionalEncoding(d_model, seq_len, dropout=0.1)
x = embed(sentense_tokens)
output = pos_enc(x)
print(output)

tensor([[[-0.3799, -3.3065,  1.8199,  7.8199, -4.3673, -0.3805,  3.0136,
           4.6391],
         [ 0.3255,  0.6746, -1.1567, -2.4967, -1.3545,  5.3980,  1.5501,
           2.2326],
         [ 0.9806,  1.6151,  0.5574,  1.4323, -1.5856,  5.2722,  0.3719,
           1.1830],
         [-1.5975,  1.3895, -5.5438,  1.1141,  5.0852,  0.5052, -0.3715,
           2.4146],
         [-1.2003,  0.0000, -2.1066, -0.0000, -2.3157,  3.6566,  4.9502,
           8.2262],
         [-0.5754,  3.7871, -5.8644, -3.2511,  3.2807, -0.0000, -0.8585,
          -2.9671]]], grad_fn=<MulBackward0>)


In [8]:
class MultiHeadAttentionBlock(nn.Module):
    def __init__(self, h: int, d_model: int, dropout: float):
        super().__init__()
        self.d_model = d_model
        self.h = h
        assert d_model % h ==0, "d_model is not divisible by h"
        self.d_k = d_model // h
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
    @staticmethod
    def attention(query, key, value, mask, dropout: nn.Dropout):
        d_k = query.shape[-1]
        attention_scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)
        if mask is not None:
            attention_scores = attention_scores.masked_fill(mask == 0, -1e9)
        attention_probs = F.softmax(attention_scores, dim = -1)
        if dropout is not None:
            attention_probs = dropout(attention_scores)
        return (attention_scores @ value, attention_probs)
    
    def forward(self, q, k, v, mask):
        query = self.w_q(q)
        key = self.w_k(k)
        value = self.w_v(v)

        query = query.view(query[0], query.shape[1], self.h, self.d_k).transpose(1,2)
        key = query.view(key[0], key.shape[1], self.h, self.d_k).transpose(1,2)
        value = value.view(value[0], value.shape[1], self.h, self.d_k).transpose(1,2)

        x, self.attention_scores = MultiHeadAttentionBlock.attention(query, key, value, mask, self.dropout )
        return self.w_o(x)




